# 🔍 PNCP — Extração e Análise de Itens: Switch

**Planilha de entrada:** `licitacoes.xlsx`  
**Total de linhas:** 68 | **Com ID PNCP válido:** 33 | **Com switch nos itens:** 32

### Fluxo
1. Lê a planilha e filtra linhas com ID PNCP válido
2. Consulta a API do PNCP para cada ID → busca os itens
3. Filtra itens onde `descricao` contém `switch`
4. Aplica **dois algoritmos** para classificar se switch é o item principal:
   - **Algoritmo 1 — Regex:** regras textuais determinísticas
   - **Algoritmo 2 — Ollama:** LLM local (classificação + extração)
5. Extrai **marca e modelo** via cada algoritmo
6. Busca resultado homologado (vencedor + preço pago)
7. Gera **relatório comparativo** e exporta Excel

## ⚙️ Célula 1 — Configurações

In [22]:
# ── Arquivo de entrada ────────────────────────────────────────
ARQUIVO_ENTRADA  = "licitacoes.xlsx"          # coloque na mesma pasta do notebook
COLUNA_ID_PNCP   = "ID PNCP"                  # nome exato da coluna

# ── Arquivos de saída ─────────────────────────────────────────
ARQUIVO_DADOS     = "switch_dados_extraidos.xlsx"
ARQUIVO_RELATORIO = "switch_relatorio_comparativo.xlsx"

# ── Ollama ────────────────────────────────────────────────────
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3"   # troque pelo modelo instalado (ex: mistral, phi3, gemma)

# ── API PNCP ──────────────────────────────────────────────────
PNCP_BASE_URL    = "https://pncp.gov.br/api/pncp/v1"
PAUSA_API_SEG    = 0.5    # pausa entre chamadas à API (evita rate limit)
PAUSA_OLLAMA_SEG = 2    # pausa entre chamadas ao Ollama

print("Configuracoes carregadas")

Configuracoes carregadas


## 📦 Célula 2 — Dependências

In [23]:
import subprocess, sys

for pkg in ["requests", "pandas", "openpyxl", "tqdm"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import re, time, json
import requests
import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import display

print("Imports OK")

Imports OK


## 🔧 Célula 3 — Parse do ID PNCP e API

In [28]:
# Formato padrão: {14 digitos}-{digitos}-{digitos}/{4 digitos}
_PATTERN_ID = re.compile(r'^(\d{14})-(\d+)-(\d+)/(\d{4})$')


def parse_id_pncp(id_pncp) -> dict:
    """
    '14226731000164-1-000018/2025' -> {'cnpj':'14226731000164', 'sequencial':000018, 'ano':2025}
    Retorna None se o formato for invalido.
    """
    if not id_pncp or str(id_pncp).strip().lower() in ('nan', ''):
        return None
    m = _PATTERN_ID.match(str(id_pncp).strip())
    if not m:
        return None
    return {
        'cnpj':       m.group(1),
        'sequencial': int(m.group(3)),   # '000018' -> 18 (grupo 2 = unidade, ignorado)
        'ano':        int(m.group(4)),
    }


def api_get(url: str):
    try:
        r = requests.get(url, headers={"accept": "*/*"}, timeout=15)
        print(f"URL: {url}")
        print(f"Status: {r.status_code}")
        print(f"Resposta: {r.text[:500]}")
        
        if r.status_code == 200:
            return r.json()
        return None
    except Exception as e:
        print(f"Erro na requisição: {e}")
        return None


def buscar_contratacao(cnpj, ano, seq):
    return api_get(f"{PNCP_BASE_URL}/v1/orgaos/{cnpj}/compras/{ano}/{seq}")


def buscar_itens(cnpj, ano, seq) -> list:
    dados = api_get(f"{PNCP_BASE_URL}/v1/orgaos/{cnpj}/compras/{ano}/{seq}/itens")
    if isinstance(dados, list):
        return dados
    if isinstance(dados, dict):
        return dados.get('data', dados.get('itens', []))
    return []


def buscar_resultados(cnpj, ano, seq, numero_item) -> list:
    dados = api_get(
        f"{PNCP_BASE_URL}/v1/orgaos/{cnpj}/compras/{ano}/{seq}/itens/{numero_item}/resultados"
    )
    return dados if isinstance(dados, list) else []


# Testes de parse
# Observacao: em Python, inteiros nao podem ter zeros a esquerda (ex: 000018).
# Como parse_id_pncp converte para int, o esperado deve usar inteiros "normais".
casos = [
    ('14226731000164-1-000018/2025',  {'cnpj': '14226731000164', 'sequencial': 18,    'ano': 2025}),
    ('00394452000103-1-017918/2025',  {'cnpj': '00394452000103', 'sequencial': 17918, 'ano': 2025}),
    ('00008.20250514/0002-24',         None),
]
print("Testes parse_id_pncp:")
for id_pncp, esperado in casos:
    obtido = parse_id_pncp(id_pncp)
    ok = 'OK' if obtido == esperado else 'FALHOU'
    print(f"  [{ok}] '{id_pncp}' -> {obtido}")

Testes parse_id_pncp:
  [OK] '14226731000164-1-000018/2025' -> {'cnpj': '14226731000164', 'sequencial': 18, 'ano': 2025}
  [OK] '00394452000103-1-017918/2025' -> {'cnpj': '00394452000103', 'sequencial': 17918, 'ano': 2025}
  [OK] '00008.20250514/0002-24' -> None


In [34]:
import requests

url = "https://pncp.gov.br/api/pncp/v1/orgaos/14226731000164/compras/2025/000018/itens"

try:
    r = requests.get(
        url,
        headers={
            "Accept": "application/json",
            "User-Agent": "Mozilla/5.0"
        },
        timeout=30
    )

    print("STATUS:", r.status_code)
    print("HEADERS:", r.headers.get("Content-Type"))
    print("TEXTO:", r.text[:500])

except Exception as e:
    print("ERRO REAL:", repr(e))

STATUS: 200
HEADERS: application/json
TEXTO: [{"numeroItem":6433348,"descricao":"MOUSE USB","materialOuServico":"M","materialOuServicoNome":"Material","valorUnitarioEstimado":11.6700,"valorTotal":175.0500,"quantidade":15.0000,"unidadeMedida":"UND","orcamentoSigiloso":false,"itemCategoriaId":3,"itemCategoriaNome":"Não se aplica","patrimonio":null,"codigoRegistroImobiliario":null,"criterioJulgamentoId":1,"criterioJulgamentoNome":"Menor preço","situacaoCompraItem":2,"situacaoCompraItemNome":"Homologado","tipoBeneficio":1,"tipoBeneficioNome":"


In [35]:
import re
from typing import Dict, List, Optional, Any
import requests


BASE_API = "https://pncp.gov.br/api/pncp/v1"

HEADERS = {
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0 (compatible; PNCP-item-parser/1.0)"
}


def ensure_session() -> requests.Session:
    session = requests.Session()
    session.headers.update(HEADERS)
    return session


def get_json(session: requests.Session, url: str, timeout: int = 30) -> Optional[Any]:
    try:
        print(f"GET {url}")
        resp = session.get(url, timeout=timeout)
        print(f"Status: {resp.status_code}")

        if resp.status_code in (204, 404):
            return None

        resp.raise_for_status()

        if not resp.text.strip():
            return None

        return resp.json()

    except Exception as e:
        print(f"Erro ao consultar {url}: {repr(e)}")
        return None


def extract_list(payload: Optional[Any]) -> List[dict]:
    if not payload:
        return []

    if isinstance(payload, list):
        return payload

    if isinstance(payload, dict):
        for key in ["data", "dados", "items", "resultado", "resultados"]:
            value = payload.get(key)
            if isinstance(value, list):
                return value

        # Caso venha um único objeto
        return [payload]

    return []


def get_itens(
    session: requests.Session,
    cnpj: str,
    ano_compra: int,
    sequencial_compra: str
) -> List[dict]:
    url = f"{BASE_API}/orgaos/{cnpj}/compras/{ano_compra}/{sequencial_compra}/itens"
    payload = get_json(session, url)
    return extract_list(payload)


def get_resultados_item(
    session: requests.Session,
    cnpj: str,
    ano_compra: int,
    sequencial_compra: str,
    numero_item: int
) -> List[dict]:
    url = f"{BASE_API}/orgaos/{cnpj}/compras/{ano_compra}/{sequencial_compra}/itens/{numero_item}/resultados"
    payload = get_json(session, url)
    return extract_list(payload)


def find_first_nonempty(d: dict, keys: List[str]) -> Optional[str]:
    for key in keys:
        value = d.get(key)
        if value is not None:
            value = str(value).strip()
            if value:
                return value
    return None


def infer_modelo_from_text(texto: Optional[str]) -> Optional[str]:
    if not texto:
        return None

    texto = str(texto).strip()

    # Heurística simples para capturar padrões comuns de modelo
    padroes = [
        r"\bmodelo[:\s\-]+([A-Za-z0-9\-_./]+)\b",
        r"\bmod[:\s\-]+([A-Za-z0-9\-_./]+)\b",
        r"\bref(?:er[eê]ncia)?[:\s\-]+([A-Za-z0-9\-_./]+)\b",
    ]

    for padrao in padroes:
        match = re.search(padrao, texto, flags=re.IGNORECASE)
        if match:
            return match.group(1).strip()

    return None


def extrair_marca_modelo_do_resultado(resultado: dict) -> Dict[str, Optional[str]]:
    """
    Tenta encontrar marca/modelo em diferentes nomes de campo.
    """
    marca = find_first_nonempty(resultado, [
        "marca",
        "nomeMarca",
        "descricaoMarca",
        "marcaProduto"
    ])

    modelo = find_first_nonempty(resultado, [
        "modelo",
        "nomeModelo",
        "descricaoModelo",
        "modeloProduto"
    ])

    # fallback por texto livre
    if not modelo:
        campos_textuais = [
            resultado.get("descricao"),
            resultado.get("descricaoDetalhada"),
            resultado.get("especificacao"),
            resultado.get("informacaoComplementar"),
            resultado.get("objetoCompra"),
        ]

        for texto in campos_textuais:
            modelo_extraido = infer_modelo_from_text(texto)
            if modelo_extraido:
                modelo = modelo_extraido
                break

    return {
        "marca": marca,
        "modelo": modelo
    }


def extrair_numero_item(item: dict) -> Optional[int]:
    candidatos = [
        item.get("numeroItem"),
        item.get("numItem"),
        item.get("item"),
        item.get("sequencialItem"),
    ]

    for valor in candidatos:
        if valor is None:
            continue
        try:
            return int(valor)
        except Exception:
            pass

    return None


def montar_relatorio_itens_com_marca_modelo(
    session: requests.Session,
    cnpj: str,
    ano_compra: int,
    sequencial_compra: str
) -> List[dict]:
    itens = get_itens(session, cnpj, ano_compra, sequencial_compra)

    if not itens:
        print("Nenhum item encontrado.")
        return []

    relatorio = []

    for item in itens:
        numero_item = extrair_numero_item(item)

        registro = {
            "numero_item": numero_item,
            "descricao_item": find_first_nonempty(item, [
                "descricao",
                "descricaoItem",
                "objetoCompra",
                "nomeItem"
            ]),
            "marca": None,
            "modelo": None,
            "resultado_encontrado": False
        }

        if numero_item is None:
            relatorio.append(registro)
            continue

        resultados = get_resultados_item(
            session=session,
            cnpj=cnpj,
            ano_compra=ano_compra,
            sequencial_compra=sequencial_compra,
            numero_item=numero_item
        )

        if resultados:
            registro["resultado_encontrado"] = True

            # pega o primeiro resultado útil
            for resultado in resultados:
                extraido = extrair_marca_modelo_do_resultado(resultado)

                if extraido["marca"] or extraido["modelo"]:
                    registro["marca"] = extraido["marca"]
                    registro["modelo"] = extraido["modelo"]
                    break

        relatorio.append(registro)

    return relatorio


if __name__ == "__main__":
    session = ensure_session()

    cnpj = "14226731000164"
    ano_compra = 2025
    sequencial_compra = "000018"

    dados = montar_relatorio_itens_com_marca_modelo(
        session=session,
        cnpj=cnpj,
        ano_compra=ano_compra,
        sequencial_compra=sequencial_compra
    )

    for linha in dados:
        print(linha)

GET https://pncp.gov.br/api/pncp/v1/orgaos/14226731000164/compras/2025/000018/itens
Status: 200
GET https://pncp.gov.br/api/pncp/v1/orgaos/14226731000164/compras/2025/000018/itens/6433348/resultados
Status: 200
GET https://pncp.gov.br/api/pncp/v1/orgaos/14226731000164/compras/2025/000018/itens/6433349/resultados
Status: 200
GET https://pncp.gov.br/api/pncp/v1/orgaos/14226731000164/compras/2025/000018/itens/6433350/resultados
Status: 200
GET https://pncp.gov.br/api/pncp/v1/orgaos/14226731000164/compras/2025/000018/itens/6433351/resultados
Status: 200
GET https://pncp.gov.br/api/pncp/v1/orgaos/14226731000164/compras/2025/000018/itens/6433352/resultados
Status: 200
GET https://pncp.gov.br/api/pncp/v1/orgaos/14226731000164/compras/2025/000018/itens/6433353/resultados
Status: 200
GET https://pncp.gov.br/api/pncp/v1/orgaos/14226731000164/compras/2025/000018/itens/6433354/resultados
Status: 200
GET https://pncp.gov.br/api/pncp/v1/orgaos/14226731000164/compras/2025/000018/itens/6433355/resulta

## 🔤 Célula 4 — Algoritmo 1: Regex

In [29]:
# Padroes que indicam switch como COMPONENTE
_COMPONENTE = [
    r'\bcom\s+switch\b',
    r'\bswitch\s+integrad\w*\b',
    r'\bintegrad\w*\s+switch\b',
    r'\binclu\w+\s+switch\b',
    r'\bkit\b.*\bswitch\b',
    r'\bconector\w*\s+switch\w*\b',
    r'\bswitch\w*\b.*\bconector\w*\b',
    r'rack.*switch',
    r'\bpatch\s+panel\b.*\bswitch\b',
    r'\bacess[oa]rios\b.*\bswitch\w*\b',
    r'\bswitch\w*\b.*\bacess[oa]rios\b',
]

# Padroes que confirmam switch como ITEM PRINCIPAL
_PRINCIPAL = [
    r'^switch\b',
    r'^modulo\s+switch\b',
    r'\bswitch\b.*\b\d+\s*portas?\b',
    r'\bswitch\b.*\b(10/100|10/100/1000|gbps|mbps)\b',
    r'\bswitch\b.*\b(gerenci[aav]|nao.gerenci|l[23]\b|poe\b|sfp\b)',
    r'\bswitch\s+\d+\s*p\b',
    r'\bswitch\s+ethernet\b',
    r'\bswitch\s+de\s+rede\b',
]

# Marcas conhecidas para fallback
_MARCAS = [
    'INTELBRAS','CISCO','HP','DELL','NETGEAR','TP-LINK','TPLINK',
    'D-LINK','DLINK','UBIQUITI','MIKROTIK','JUNIPER','HUAWEI',
    'EXTREME','3COM','ARUBA','ZYXEL','LINKSYS','SCHNEIDER','MOXA',
]


def regex_eh_principal(descricao: str) -> bool:
    t = descricao.lower().strip()
    if 'switch' not in t:
        return False
    for p in _COMPONENTE:
        if re.search(p, t, re.IGNORECASE):
            return False
    for p in _PRINCIPAL:
        if re.search(p, t, re.IGNORECASE):
            return True
    palavras = t.split()
    return len(palavras) > 0 and palavras[0] == 'switch'


def regex_marca_modelo(descricao: str) -> dict:
    res = {'marca_regex': None, 'modelo_regex': None}
    t = descricao.strip()

    m = re.search(r'marca[:\s]+([A-Z][A-Z0-9\s\-\.]+?)(?:\s+modelo|\s*[;\/,]|$)',
                  t, re.IGNORECASE)
    if m:
        res['marca_regex'] = m.group(1).strip()

    m = re.search(r'modelo[:\s]+([A-Z0-9][A-Z0-9\s\-\/\.]+?)(?:\s*[;\/,]|\s+[A-Z]{4,}|$)',
                  t, re.IGNORECASE)
    if m:
        res['modelo_regex'] = m.group(1).strip()

    if not res['marca_regex']:
        for marca in _MARCAS:
            if re.search(r'\b' + re.escape(marca) + r'\b', t, re.IGNORECASE):
                res['marca_regex'] = marca
                break

    if res['marca_regex'] and not res['modelo_regex']:
        m = re.search(
            re.escape(res['marca_regex']) + r'\s+([A-Z0-9][A-Z0-9\-\.]{2,15})',
            t, re.IGNORECASE
        )
        if m:
            res['modelo_regex'] = m.group(1).strip()

    return res


# Testes automaticos
testes = [
    ("SWITCH 08 PORTAS 10/100/1000 MBPS",                              True),
    ("SWITCH 16 PORTAS 10/100/1000 MBPS",                              True),
    ("SWITCH GERENCIAVEL L2 24 PORTAS INTELBRAS SG 2404 QR",           True),
    ("MODULO SWITCH ETHERNET NAO GERENCIAVEL 5 PORTAS 24VCC",          True),
    ("SWITCH 8 PORTAS OITO PORTAS FAST ETHERNET LAN 10/100",           True),
    ("ROTEADOR 1200 MBPS COM SWITCH INTEGRADO 4 PORTAS",               False),
    ("KIT RACK COM PATCH PANEL E SWITCH",                              False),
    ("ACESSORIOS PARA INSTALACAO DE CABOS E CONECTORES SWITCHS",       False),
    ("MOUSE USB",                                                       False),
]
acertos = 0
print(f"{'Descricao':<56} {'Esp':>5} {'Obt':>5}")
print('-' * 70)
for desc, esp in testes:
    obt = regex_eh_principal(desc)
    ok  = 'OK' if obt == esp else 'FALHOU'
    if obt == esp: acertos += 1
    print(f"{desc[:55]:<56} {str(esp):>5} {str(obt):>5}  [{ok}]")
print(f"\nAcuracia nos testes: {acertos}/{len(testes)} ({100*acertos/len(testes):.0f}%)")

Descricao                                                  Esp   Obt
----------------------------------------------------------------------
SWITCH 08 PORTAS 10/100/1000 MBPS                         True  True  [OK]
SWITCH 16 PORTAS 10/100/1000 MBPS                         True  True  [OK]
SWITCH GERENCIAVEL L2 24 PORTAS INTELBRAS SG 2404 QR      True  True  [OK]
MODULO SWITCH ETHERNET NAO GERENCIAVEL 5 PORTAS 24VCC     True  True  [OK]
SWITCH 8 PORTAS OITO PORTAS FAST ETHERNET LAN 10/100      True  True  [OK]
ROTEADOR 1200 MBPS COM SWITCH INTEGRADO 4 PORTAS         False False  [OK]
KIT RACK COM PATCH PANEL E SWITCH                        False False  [OK]
ACESSORIOS PARA INSTALACAO DE CABOS E CONECTORES SWITCH  False False  [OK]
MOUSE USB                                                False False  [OK]

Acuracia nos testes: 9/9 (100%)


## 🤖 Célula 5 — Algoritmo 2: Ollama

In [30]:
def verificar_ollama() -> bool:
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        if r.status_code == 200:
            modelos = [m['name'] for m in r.json().get('models', [])]
            print(f"Ollama online. Modelos: {modelos}")
            if not any(OLLAMA_MODEL in m for m in modelos):
                print(f"AVISO: Modelo '{OLLAMA_MODEL}' nao encontrado. Ajuste OLLAMA_MODEL na Celula 1.")
                return False
            return True
    except Exception:
        pass
    print("Ollama offline. Rode: ollama serve")
    return False


_PROMPT = """Voce eh especialista em licitacoes publicas brasileiras de TI.
Analise a descricao de um item de licitacao.

Responda APENAS com JSON valido, sem texto extra, sem markdown:
{{"principal": true/false, "marca": "string ou null", "modelo": "string ou null"}}

Regras:
- "principal": true SOMENTE se SWITCH for o produto sendo licitado (ex: SWITCH 8 PORTAS).
  false se switch for componente, acessorio ou parte de kit/conjunto.
- "marca": fabricante mencionado (ex: INTELBRAS, CISCO) ou null.
- "modelo": codigo do modelo (ex: SG 800Q) ou null.

Descricao: {descricao}"""


def ollama_classificar(descricao: str) -> dict:
    vazio = {'eh_principal_ollama': None, 'marca_ollama': None,
             'modelo_ollama': None, 'erro_ollama': None}
    try:
        r = requests.post(
            OLLAMA_URL,
            json={"model": OLLAMA_MODEL,
                  "prompt": _PROMPT.format(descricao=descricao),
                  "stream": False},
            timeout=45
        )
        if r.status_code != 200:
            return {**vazio, 'erro_ollama': f'HTTP {r.status_code}'}

        texto = re.sub(r'```json|```', '', r.json().get('response', '')).strip()
        match = re.search(r'\{.*?\}', texto, re.DOTALL)
        if not match:
            return {**vazio, 'erro_ollama': f'JSON nao encontrado: {texto[:80]}'}

        dados = json.loads(match.group())
        return {
            'eh_principal_ollama': bool(dados.get('principal')),
            'marca_ollama':        dados.get('marca') or None,
            'modelo_ollama':       dados.get('modelo') or None,
            'erro_ollama':         None,
        }
    except json.JSONDecodeError as e:
        return {**vazio, 'erro_ollama': f'JSON invalido: {e}'}
    except Exception as e:
        return {**vazio, 'erro_ollama': str(e)}


OLLAMA_OK = verificar_ollama()

if OLLAMA_OK:
    print("\nTeste rapido Ollama:")
    for desc in [
        "SWITCH 08 PORTAS 10/100/1000 MBPS INTELBRAS SG 800Q",
        "ACESSORIOS PARA INSTALACAO DE CABOS E CONECTORES SWITCHS",
    ]:
        res = ollama_classificar(desc)
        print(f"  '{desc[:55]}' -> principal={res['eh_principal_ollama']} marca={res['marca_ollama']}")

Ollama online. Modelos: ['qwen2.5:14b', 'deepseek-r1:7b', 'qwen2.5:7b', 'phi3:mini', 'llama3:70b', 'llama3:8b', 'llama3:latest', 'llama3.2:1b', 'mistral:7b-instruct-q4_0', 'llama2:latest', 'llama3:8b-instruct-q4_0']

Teste rapido Ollama:
  'SWITCH 08 PORTAS 10/100/1000 MBPS INTELBRAS SG 800Q' -> principal=True marca=INTELBRAS
  'ACESSORIOS PARA INSTALACAO DE CABOS E CONECTORES SWITCH' -> principal=False marca=None


## 📥 Célula 6 — Carrega a Planilha

In [31]:
if ARQUIVO_ENTRADA.endswith('.csv'):
    df_entrada = pd.read_csv(ARQUIVO_ENTRADA, dtype=str)
else:
    df_entrada = pd.read_excel(ARQUIVO_ENTRADA, dtype=str)

df_entrada = df_entrada.reset_index(drop=True)
df_entrada['_params'] = df_entrada[COLUNA_ID_PNCP].apply(parse_id_pncp)
df_validos = df_entrada[df_entrada['_params'].notna()].copy()
df_invalidos = df_entrada[df_entrada['_params'].isna()]

print(f"Total de linhas:              {len(df_entrada)}")
print(f"Com ID PNCP valido:           {len(df_validos)}")
print(f"Ignorados (formato invalido): {len(df_invalidos)}")
if len(df_invalidos) > 0:
    print(f"  Exemplos: {df_invalidos[COLUNA_ID_PNCP].tolist()[:5]}")

Total de linhas:              68
Com ID PNCP valido:           33
Ignorados (formato invalido): 35
  Exemplos: [nan, nan, nan, nan, nan]


## 🚀 Célula 7 — Pipeline Principal

In [32]:
resultados = []
erros      = []

for _, row in tqdm(df_validos.iterrows(), total=len(df_validos), desc="Processando licitacoes"):
    id_pncp = str(row[COLUNA_ID_PNCP]).strip()
    params  = row['_params']
    cnpj, ano, seq = params['cnpj'], params['ano'], params['sequencial']

    # 1. Dados gerais da contratacao
    contratacao = buscar_contratacao(cnpj, ano, seq)
    time.sleep(PAUSA_API_SEG)
    if not contratacao:
        erros.append({'id_pncp': id_pncp, 'erro': 'Contratacao nao encontrada na API'})
        continue

    # 2. Itens da contratacao
    itens = buscar_itens(cnpj, ano, seq)
    time.sleep(PAUSA_API_SEG)
    if not itens:
        erros.append({'id_pncp': id_pncp, 'erro': 'Nenhum item retornado'})
        continue

    # 3. Filtra itens com 'switch' na descricao
    itens_switch = [
        item for item in itens
        if 'switch' in str(item.get('descricao', '')).lower()
    ]
    if not itens_switch:
        continue

    # 4. Processa cada item com switch
    for item in itens_switch:
        descricao   = str(item.get('descricao', ''))
        numero_item = item.get('numeroItem')

        # Algoritmo 1 — Regex
        eh_principal_regex = regex_eh_principal(descricao)
        extr_regex         = regex_marca_modelo(descricao)

        # Algoritmo 2 — Ollama
        if OLLAMA_OK:
            extr_ollama = ollama_classificar(descricao)
            time.sleep(PAUSA_OLLAMA_SEG)
        else:
            extr_ollama = {
                'eh_principal_ollama': None, 'marca_ollama': None,
                'modelo_ollama': None, 'erro_ollama': 'Ollama offline'
            }

        # 5. Resultado homologado
        resultados_item = buscar_resultados(cnpj, ano, seq, numero_item)
        time.sleep(PAUSA_API_SEG)
        vencedor = next(
            (r for r in resultados_item if r.get('ordemClassificacaoSrp') == 1),
            resultados_item[0] if resultados_item else {}
        )

        resultados.append({
            # Identificacao
            'id_pncp':              id_pncp,
            'cnpj_orgao':           cnpj,
            'ano':                  ano,
            'sequencial':           seq,
            'numero_item':          numero_item,
            # Contratacao
            'orgao':                contratacao.get('orgaoEntidade', {}).get('razaoSocial', ''),
            'municipio':            contratacao.get('unidadeOrgao', {}).get('municipioNome', ''),
            'uf':                   contratacao.get('unidadeOrgao', {}).get('ufSigla', ''),
            'modalidade':           contratacao.get('modalidadeNome', ''),
            'situacao_contratacao': contratacao.get('situacaoCompraNome', ''),
            'data_abertura':        contratacao.get('dataAberturaProposta', ''),
            'data_publicacao':      contratacao.get('dataPublicacaoPncp', ''),
            'srp':                  contratacao.get('srp', ''),
            # Item
            'descricao':            descricao,
            'material_servico':     item.get('materialOuServicoNome', ''),
            'quantidade':           item.get('quantidade', ''),
            'unidade_medida':       item.get('unidadeMedida', ''),
            'valor_unit_estimado':  item.get('valorUnitarioEstimado', ''),
            'valor_total_estimado': item.get('valorTotal', ''),
            'situacao_item':        item.get('situacaoCompraItemNome', ''),
            'tem_resultado':        item.get('temResultado', ''),
            'ncm':                  item.get('ncmNbsCodigo', ''),
            # Vencedor
            'cnpj_vencedor':        vencedor.get('niFornecedor', ''),
            'nome_vencedor':        vencedor.get('nomeRazaoSocialFornecedor', ''),
            'porte_vencedor':       vencedor.get('porteFornecedorNome', ''),
            'valor_unit_homolog':   vencedor.get('valorUnitarioHomologado', ''),
            'valor_total_homolog':  vencedor.get('valorTotalHomologado', ''),
            'qtd_homologada':       vencedor.get('quantidadeHomologada', ''),
            'data_resultado':       vencedor.get('dataResultado', ''),
            # Algoritmo 1 — Regex
            'principal_regex':  eh_principal_regex,
            'marca_regex':      extr_regex.get('marca_regex'),
            'modelo_regex':     extr_regex.get('modelo_regex'),
            # Algoritmo 2 — Ollama
            'principal_ollama': extr_ollama.get('eh_principal_ollama'),
            'marca_ollama':     extr_ollama.get('marca_ollama'),
            'modelo_ollama':    extr_ollama.get('modelo_ollama'),
            'erro_ollama':      extr_ollama.get('erro_ollama'),
        })

print(f"\nConcluido!")
print(f"  Itens com 'switch' encontrados: {len(resultados)}")
print(f"  Erros de API:                   {len(erros)}")

Processando licitacoes:   0%|          | 0/33 [00:00<?, ?it/s]

URL: https://pncp.gov.br/api/pncp/v1/v1/orgaos/14226731000164/compras/2025/18
Status: 404
Resposta: {"timestamp":"2026-03-20T18:18:18.390+00:00","status":404,"error":"Not Found","message":"No message available","path":"/pncp-api/v1/v1/orgaos/14226731000164/compras/2025/18"}
URL: https://pncp.gov.br/api/pncp/v1/v1/orgaos/18338285000130/compras/2025/79
Status: 404
Resposta: {"timestamp":"2026-03-20T18:18:19.015+00:00","status":404,"error":"Not Found","message":"No message available","path":"/pncp-api/v1/v1/orgaos/18338285000130/compras/2025/79"}
URL: https://pncp.gov.br/api/pncp/v1/v1/orgaos/58290502000184/compras/2025/93
Status: 404
Resposta: {"timestamp":"2026-03-20T18:18:19.699+00:00","status":404,"error":"Not Found","message":"No message available","path":"/pncp-api/v1/v1/orgaos/58290502000184/compras/2025/93"}
URL: https://pncp.gov.br/api/pncp/v1/v1/orgaos/46377800000127/compras/2025/4081
Status: 404
Resposta: {"timestamp":"2026-03-20T18:18:20.316+00:00","status":404,"error":"Not Fo

## 📊 Célula 8 — Análise Comparativa

In [33]:
df = pd.DataFrame(resultados)

if df.empty:
    print("Nenhum item com 'switch' encontrado na API.")
    print("Isso pode ocorrer porque as licitacoes ainda nao tem itens cadastrados no PNCP.")
else:
    n = len(df)
    n_regex  = int(df['principal_regex'].sum())
    n_ollama = int(df['principal_ollama'].sum()) if (
        OLLAMA_OK and 'principal_ollama' in df.columns
        and df['principal_ollama'].notna().any()
    ) else None

    SEP = '=' * 52
    print(SEP)
    print(f" ITENS COM 'SWITCH' NA API: {n}")
    print(SEP)

    print(f"\nCLASSIFICACAO — switch como item PRINCIPAL")
    print(f"  Regex:  {n_regex:>3}/{n}  ({100*n_regex/n:.1f}%)")
    if n_ollama is not None:
        print(f"  Ollama: {n_ollama:>3}/{n}  ({100*n_ollama/n:.1f}%)")
    else:
        print(f"  Ollama: nao disponivel")

    # Concordancia
    if OLLAMA_OK and 'principal_ollama' in df.columns:
        df_c = df.dropna(subset=['principal_ollama'])
        if len(df_c) > 0:
            conc  = (df_c['principal_regex'] == df_c['principal_ollama']).mean()
            n_div = int((df_c['principal_regex'] != df_c['principal_ollama']).sum())
            print(f"\nCONCORDANCIA ENTRE ALGORITMOS")
            print(f"  Taxa: {conc*100:.1f}%  |  Divergencias: {n_div} itens")
            if n_div > 0:
                print("\nItens divergentes:")
                div = df_c[df_c['principal_regex'] != df_c['principal_ollama']][
                    ['descricao','principal_regex','principal_ollama']
                ].reset_index(drop=True)
                display(div)

    # Switch principal (Regex)
    print(f"\n{SEP}")
    print(" SWITCH PRINCIPAL — Regex")
    print(SEP)
    df_p = df[df['principal_regex'] == True].copy()
    for col in ['valor_unit_estimado','valor_unit_homolog']:
        df_p[col] = pd.to_numeric(df_p[col], errors='coerce')
    display(df_p[['descricao','quantidade','valor_unit_estimado',
                  'valor_unit_homolog','nome_vencedor',
                  'marca_regex','modelo_regex']].reset_index(drop=True))

    # Estatisticas de preco
    df_prec = df_p.dropna(subset=['valor_unit_homolog'])
    if len(df_prec) > 0:
        s = df_prec['valor_unit_homolog']
        print(f"\nPRECOS — Switch Principal (homologados)")
        print(f"  Amostras: {len(df_prec)}")
        print(f"  Minimo:   R$ {s.min():>10,.2f}")
        print(f"  Maximo:   R$ {s.max():>10,.2f}")
        print(f"  Media:    R$ {s.mean():>10,.2f}")
        print(f"  Mediana:  R$ {s.median():>10,.2f}")

        df_d = df_prec.dropna(subset=['valor_unit_estimado'])
        df_d = df_d[df_d['valor_unit_estimado'] > 0].copy()
        if len(df_d) > 0:
            df_d['desc_pct'] = (
                (df_d['valor_unit_estimado'] - df_d['valor_unit_homolog'])
                / df_d['valor_unit_estimado'] * 100
            )
            print(f"  Desconto medio (est->homolog): {df_d['desc_pct'].mean():.1f}%")

Nenhum item com 'switch' encontrado na API.
Isso pode ocorrer porque as licitacoes ainda nao tem itens cadastrados no PNCP.


## 💾 Célula 9 — Exportação Excel

In [ ]:
if df.empty:
    print("Nada para exportar.")
else:
    # Arquivo 1: dados brutos
    df.to_excel(ARQUIVO_DADOS, index=False)
    print(f"Exportado: {ARQUIVO_DADOS}  ({len(df)} linhas)")

    # Arquivo 2: relatorio com abas
    COLUNAS = [
        'id_pncp','orgao','municipio','uf','descricao',
        'quantidade','unidade_medida',
        'valor_unit_estimado','valor_unit_homolog','valor_total_homolog',
        'nome_vencedor','cnpj_vencedor','porte_vencedor',
        'data_resultado','situacao_item','srp',
        'principal_regex','marca_regex','modelo_regex',
        'principal_ollama','marca_ollama','modelo_ollama',
    ]
    cols = [c for c in COLUNAS if c in df.columns]

    with pd.ExcelWriter(ARQUIVO_RELATORIO, engine='openpyxl') as writer:

        # Aba 1: todos os itens com switch
        df[cols].to_excel(writer, sheet_name='Todos Switch', index=False)

        # Aba 2: switch principal (Regex)
        df_regex = df[df['principal_regex'] == True]
        df_regex[cols].to_excel(writer, sheet_name='Principal (Regex)', index=False)

        # Aba 3: switch principal (Ollama)
        if OLLAMA_OK and 'principal_ollama' in df.columns:
            df_oll = df[df['principal_ollama'] == True]
            df_oll[cols].to_excel(writer, sheet_name='Principal (Ollama)', index=False)

        # Aba 4: divergencias
        if OLLAMA_OK and 'principal_ollama' in df.columns:
            df_div = df[df['principal_regex'] != df['principal_ollama']]
            if len(df_div) > 0:
                df_div[['id_pncp','descricao','principal_regex',
                         'principal_ollama','erro_ollama']].to_excel(
                    writer, sheet_name='Divergencias', index=False)

        # Aba 5: comparativo de precos
        df_p = df[df['principal_regex'] == True].copy()
        for col in ['valor_unit_estimado','valor_unit_homolog']:
            df_p[col] = pd.to_numeric(df_p[col], errors='coerce')
        df_p = df_p.dropna(subset=['valor_unit_homolog']).copy()
        if len(df_p) > 0:
            df_p['desconto_%'] = (
                (df_p['valor_unit_estimado'] - df_p['valor_unit_homolog'])
                / df_p['valor_unit_estimado'].replace(0, float('nan')) * 100
            ).round(2)
            cols_p = ['id_pncp','orgao','municipio','uf','descricao',
                      'marca_regex','modelo_regex','quantidade',
                      'valor_unit_estimado','valor_unit_homolog','desconto_%',
                      'nome_vencedor','cnpj_vencedor','data_resultado']
            cols_p = [c for c in cols_p if c in df_p.columns]
            df_p.sort_values('valor_unit_homolog')[cols_p].to_excel(
                writer, sheet_name='Comparativo Precos', index=False)

        # Aba 6: erros
        if erros:
            pd.DataFrame(erros).to_excel(writer, sheet_name='Erros API', index=False)

    print(f"Exportado: {ARQUIVO_RELATORIO}")
    print()
    print("Abas:")
    print("  Todos Switch        — todos os itens onde 'switch' aparece na descricao")
    print("  Principal (Regex)   — switch como item principal (Algoritmo 1)")
    if OLLAMA_OK:
        print("  Principal (Ollama)  — switch como item principal (Algoritmo 2)")
        print("  Divergencias        — casos onde os dois algoritmos discordam")
    print("  Comparativo Precos  — precos ordenados do menor ao maior")
    if erros:
        print("  Erros API           — IDs que falharam na consulta")

Nada para exportar.
